# CNN TensorFlow 2.0
本 notebook 采用纯 TF2 ，基于 `tf.keras` 和 `tf.data`，不使用 `tf.placeholder` 和 `tf.Session`。

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers

print('TF version', tf.__version__)

# 定义 conv2d、max_pool_2x2 TF2 函数

def conv2d(x, W):
    return tf.nn.conv2d(x, W, strides=[1, 1, 1, 1], padding='SAME')


def max_pool_2x2(x):
    return tf.nn.max_pool2d(x, ksize=2, strides=2, padding='SAME')

# 加载 MNIST
(train_images, train_labels), (test_images, test_labels) = tf.keras.datasets.mnist.load_data(path='mnist.npz')
train_images = train_images.astype('float32') / 255.0
test_images = test_images.astype('float32') / 255.0
train_images = train_images[..., tf.newaxis]
test_images = test_images[..., tf.newaxis]

num_classes = 10
train_labels = tf.keras.utils.to_categorical(train_labels, num_classes)
test_labels = tf.keras.utils.to_categorical(test_labels, num_classes)

batch_size = 100
train_ds = tf.data.Dataset.from_tensor_slices((train_images, train_labels)).shuffle(60000).batch(batch_size)
test_ds = tf.data.Dataset.from_tensor_slices((test_images, test_labels)).batch(batch_size)

# 用低层 API 构建与原框架相等价模型（两层卷积 + 两个池化）
W_conv1 = tf.Variable(tf.random.truncated_normal([7, 7, 1, 32], stddev=0.1), name='W_conv1')
b_conv1 = tf.Variable(tf.constant(0.1, shape=[32]), name='b_conv1')
W_conv2 = tf.Variable(tf.random.truncated_normal([5, 5, 32, 64], stddev=0.1), name='W_conv2')
b_conv2 = tf.Variable(tf.constant(0.1, shape=[64]), name='b_conv2')
W_fc1 = tf.Variable(tf.random.truncated_normal([7 * 7 * 64, 1024], stddev=0.1), name='W_fc1')
b_fc1 = tf.Variable(tf.constant(0.1, shape=[1024]), name='b_fc1')
W_fc2 = tf.Variable(tf.random.truncated_normal([1024, 10], stddev=0.1), name='W_fc2')
b_fc2 = tf.Variable(tf.constant(0.1, shape=[10]), name='b_fc2')

keep_prob_rate = 0.7
learning_rate = 1e-4
max_epoch = 20
optimizer = optimizers.Adam(learning_rate=learning_rate)


def forward(x, training=False):
    x = tf.reshape(x, [-1, 28, 28, 1])
    h_conv1 = tf.nn.relu(conv2d(x, W_conv1) + b_conv1)
    h_pool1 = max_pool_2x2(h_conv1)
    h_conv2 = tf.nn.relu(conv2d(h_pool1, W_conv2) + b_conv2)
    h_pool2 = max_pool_2x2(h_conv2)
    h_pool2_flat = tf.reshape(h_pool2, [-1, 7 * 7 * 64])
    h_fc1 = tf.nn.relu(tf.matmul(h_pool2_flat, W_fc1) + b_fc1)
    if training:
        h_fc1 = tf.nn.dropout(h_fc1, 1 - keep_prob_rate)
    logits = tf.matmul(h_fc1, W_fc2) + b_fc2
    return logits


loss_fn = tf.keras.losses.CategoricalCrossentropy(from_logits=True)


@tf.function
def train_step(images, labels):
    with tf.GradientTape() as tape:
        logits = forward(images, training=True)
        loss = loss_fn(labels, logits)
    grads = tape.gradient(loss, [W_conv1, b_conv1, W_conv2, b_conv2, W_fc1, b_fc1, W_fc2, b_fc2])
    optimizer.apply_gradients(zip(grads, [W_conv1, b_conv1, W_conv2, b_conv2, W_fc1, b_fc1, W_fc2, b_fc2]))
    preds = tf.nn.softmax(logits)
    acc = tf.reduce_mean(tf.cast(tf.equal(tf.argmax(preds, axis=1), tf.argmax(labels, axis=1)), tf.float32))
    return loss, acc


@tf.function
def test_step(images, labels):
    logits = forward(images, training=False)
    loss = loss_fn(labels, logits)
    preds = tf.nn.softmax(logits)
    acc = tf.reduce_mean(tf.cast(tf.equal(tf.argmax(preds, axis=1), tf.argmax(labels, axis=1)), tf.float32))
    return loss, acc

for epoch in range(max_epoch):
    total_loss = 0.0
    total_acc = 0.0
    batches = 0
    for x_batch, y_batch in train_ds:
        l, a = train_step(x_batch, y_batch)
        total_loss += l
        total_acc += a
        batches += 1
    print(f'Epoch {epoch + 1}/{max_epoch}: loss={total_loss / batches:.4f}, acc={total_acc / batches:.4f}')

# 测试

t_loss = 0.0
t_acc = 0.0
t_batches = 0
for x_batch, y_batch in test_ds:
    l, a = test_step(x_batch, y_batch)
    t_loss += l
    t_acc += a
    t_batches += 1
print(f'Test loss={t_loss / t_batches:.4f}, Test acc={t_acc / t_batches:.4f}')

TF version 2.10.0
Epoch 1/20: loss=0.6122, acc=0.8506
Epoch 2/20: loss=0.1315, acc=0.9595
Epoch 3/20: loss=0.0856, acc=0.9731
Epoch 4/20: loss=0.0663, acc=0.9793
Epoch 5/20: loss=0.0507, acc=0.9838
Epoch 6/20: loss=0.0406, acc=0.9870
Epoch 7/20: loss=0.0334, acc=0.9893
Epoch 8/20: loss=0.0294, acc=0.9907
Epoch 9/20: loss=0.0245, acc=0.9923
Epoch 10/20: loss=0.0198, acc=0.9937
Epoch 11/20: loss=0.0166, acc=0.9949
Epoch 12/20: loss=0.0154, acc=0.9952
Epoch 13/20: loss=0.0121, acc=0.9967
Epoch 14/20: loss=0.0105, acc=0.9968
Epoch 15/20: loss=0.0093, acc=0.9971
Epoch 16/20: loss=0.0087, acc=0.9972
Epoch 17/20: loss=0.0070, acc=0.9977
Epoch 18/20: loss=0.0059, acc=0.9981
Epoch 19/20: loss=0.0058, acc=0.9982
Epoch 20/20: loss=0.0056, acc=0.9981
Test loss=0.0271, Test acc=0.9912
